# Epoch Spring Camp 2026 - Take Home Task 3

In this task, you will build and compare two recommender system models:

- **Matrix Factorization (MF)** using a dot product of embeddings  
- **Neural Collaborative Filtering (MLP-based)** using a multi-layer perceptron  

You are provided with:
- Preprocessed interaction data
- Evaluation pipeline

You are expected to implement:
- Negative sampling
- Model architectures
- Training loop

---

The purpose of this task is to understand how neural networks can model user–item interactions beyond simple similarity.

## Imports

In [49]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

## Loading Data

We begin by loading the interaction data.

In [50]:
df = pd.read_csv("interactions.csv")  # columns: user_id, movie_id

print("Num users:", df['user_id'].nunique())
print("Num items:", df['item_id'].nunique())
print("Num interactions:", len(df))

Num users: 942
Num items: 1447
Num interactions: 55375


Each row represents a **positive interaction** (implicit feedback).

## Train / Test Split

We split the data into training and testing sets.

The model will learn from the training data and be evaluated on unseen interactions in the test set.

In [51]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [52]:
df['user_id'].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

## Negative Sampling

The dataset only contains **positive interactions** (user interacted with item).

However, to train a model, we also need **negative examples** (user did not interact with item).

---

### Why do we need this?

We treat recommendation as a **binary classification problem**:

- Positive interaction → label = 1  
- No interaction → label = 0  

Since we don’t have explicit negatives, we **sample them**.

---

### Your Task

For each `(user, item)` interaction:
- Keep it as a **positive sample (label = 1)**
- Sample one or more items the user has **not interacted with**
  - Add them as **negative samples (label = 0)**

---

### Hints

- You can randomly sample items
- Make sure sampled negatives are **not already in the dataset**
- You may use a `set` of `(user, item)` pairs for fast lookup

---

Implement a function:
`sample_negatives(df, num_neg=1)`
that returns a dataframe with columns:
`[user, item, label]`

In [53]:
def sample_negatives(df, num_neg=1):
    # TODO:
    # 1. Create a set of (user, item) interactions
    # 2. For each positive interaction:
    #    - add (user, item, 1)
    #    - sample 'num_neg' negative items
    # 3. Return a dataframe with columns: user, item, label
    df_neg=pd.DataFrame(columns=['user_id', 'item_id', 'label'])
    interactions = set(zip(df['user_id'], df['item_id']))
    for user, item in interactions:
        df_neg = pd.concat([df_neg, pd.DataFrame({'user_id': user, 'item_id': item, 'label': 1}, index=[0])], ignore_index=True)

    for user in df['user_id'].unique():
      for _ in range(num_neg):
              neg_item = np.random.choice(df['item_id'].unique())
              while (user, neg_item) in interactions:
                  neg_item = np.random.choice(df['item_id'].unique())
              df_neg = pd.concat([df_neg, pd.DataFrame({'user_id': user, 'item_id': neg_item, 'label': 0}, index=[0])], ignore_index=True)
    return df_neg


## PyTorch Dataset

We now wrap our processed data into a PyTorch `Dataset`.

This allows us to:
- Access individual samples as `(user, item, label)`
- Easily plug into a `DataLoader` for batching

You do not need to modify this part.

In [54]:
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_id'].values.astype(np.int32))
        self.items = torch.tensor(df['item_id'].values.astype(np.int32))
        self.labels = torch.tensor(df['label'].values.astype(np.int32)).float()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

## DataLoader

The `DataLoader` will:
- Batch the data
- Shuffle the training data
- Feed it to the model during training

In [55]:
train_data = sample_negatives(train_df,num_neg=100)

train_ds,valid_ds=train_test_split(train_data,test_size=0.1,random_state=42)
train_loader = DataLoader(InteractionDataset(train_ds), batch_size=256, shuffle=True)
valid_loader = DataLoader(InteractionDataset(valid_ds), batch_size=256, shuffle=True)

Why are we sampling negatives only for the training data?

## Quick Exploration

Before building models, take a moment to explore the data.

Try to understand:
- How many interactions each user has
- How popular certain items are

This can give intuition about the dataset.

In [56]:
# Interactions per user
user_counts = df['user_id'].value_counts()
print(user_counts.describe())

# Interactions per item
item_counts = df['item_id'].value_counts()
print(item_counts.describe())

count    942.000000
mean      58.784501
std       54.696664
min        3.000000
25%       19.000000
50%       39.500000
75%       80.750000
max      378.000000
Name: count, dtype: float64
count    1447.000000
mean       38.268832
std        57.956847
min         1.000000
25%         3.000000
50%        13.000000
75%        47.500000
max       501.000000
Name: count, dtype: float64


## Baseline Model: Matrix Factorization (MF)

We represent:
- Each **user** as a vector (embedding)
- Each **item** as a vector (embedding)

To predict interaction:
- Compute the **dot product** between user and item embeddings

This gives a **score** indicating how likely the user is to interact with the item.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Computes their dot product as the output score

In [57]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        # - Define item embedding layer
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

    def forward(self, user, item):
        # TODO:
        # - Get user and item embeddings
        # - Compute dot product
        # - Return a single score per pair
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)
        return (user_emb * item_emb).sum(1)

## Training the MF Model

Now train your Matrix Factorization model.

You will need to:
- Define a loss function
- Define an optimizer
- Iterate over the DataLoader
- Update model parameters

---

### Hints

- Use **Binary Cross Entropy loss**
- Apply **sigmoid** to model outputs if needed
- Typical steps:
  - forward pass
  - compute loss
  - backward pass
  - optimizer step

In [58]:
def train(model, dataloader, val_dataloader, epochs, patience=100):
    if torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    early_stop_counter = 0
    losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for user, item, label in dataloader:
            optimizer.zero_grad()
            relevance_scores = model(user.to(device), item.to(device))
            label = label.to(device).float()

            loss = criterion(relevance_scores, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(dataloader)
        losses.append(avg_train_loss)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for user, item, label in val_dataloader:
                relevance_scores = model(user.to(device), item.to(device))
                label = label.to(device).float()
                loss = criterion(relevance_scores, label)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_dataloader)
        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

        # Scheduler step
        scheduler.step(avg_val_loss)

        # Early Stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0
            # Optional: Save best model
            best_model_state = model.state_dict()
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print("Early stopping triggered.")
                model.load_state_dict(best_model_state)
                break

    return losses

In [59]:
# TODO:
# 1. Initialize MF model
MF_model = MF(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=16)
# 2. Train it
train(MF_model, train_loader,valid_loader, epochs=300)

Epoch 1: Train Loss: 1.7009, Val Loss: 1.6463
Epoch 2: Train Loss: 1.5772, Val Loss: 1.5621
Epoch 3: Train Loss: 1.4650, Val Loss: 1.4888
Epoch 4: Train Loss: 1.3641, Val Loss: 1.4187
Epoch 5: Train Loss: 1.2735, Val Loss: 1.3525
Epoch 6: Train Loss: 1.1923, Val Loss: 1.2949
Epoch 7: Train Loss: 1.1194, Val Loss: 1.2514
Epoch 8: Train Loss: 1.0540, Val Loss: 1.1978
Epoch 9: Train Loss: 0.9954, Val Loss: 1.1406
Epoch 10: Train Loss: 0.9429, Val Loss: 1.1110
Epoch 11: Train Loss: 0.8956, Val Loss: 1.0710
Epoch 12: Train Loss: 0.8529, Val Loss: 1.0398
Epoch 13: Train Loss: 0.8141, Val Loss: 1.0072
Epoch 14: Train Loss: 0.7785, Val Loss: 0.9804
Epoch 15: Train Loss: 0.7454, Val Loss: 0.9463
Epoch 16: Train Loss: 0.7142, Val Loss: 0.9211
Epoch 17: Train Loss: 0.6842, Val Loss: 0.8962
Epoch 18: Train Loss: 0.6548, Val Loss: 0.8630
Epoch 19: Train Loss: 0.6256, Val Loss: 0.8328
Epoch 20: Train Loss: 0.5962, Val Loss: 0.8077
Epoch 21: Train Loss: 0.5668, Val Loss: 0.7699
Epoch 22: Train Loss: 

[1.7009415428251702,
 1.577218824098732,
 1.4649722245684396,
 1.3641132995088487,
 1.273521130579453,
 1.1923042364433805,
 1.1193625646694974,
 1.0540268477473171,
 0.9954325058621792,
 0.9428675913957599,
 0.8955808060369942,
 0.8528979998349654,
 0.8141095050796101,
 0.7785191582458465,
 0.7454262151855218,
 0.7142097699079181,
 0.6841973919398486,
 0.654803968552936,
 0.6255715237261089,
 0.5962438132973422,
 0.5667548540069337,
 0.5374033321345366,
 0.5086197323745281,
 0.48106473629234753,
 0.45528210690378895,
 0.4316778305374866,
 0.41050001978874207,
 0.3917158181172866,
 0.3751142477842327,
 0.360451954835739,
 0.347465732136791,
 0.3358448218148837,
 0.3254054305000227,
 0.31597840663588755,
 0.3073846324513336,
 0.2995156637513417,
 0.292282663117444,
 0.28560750605634105,
 0.27938466633859355,
 0.2735906366939662,
 0.26810622946560014,
 0.26294568355445747,
 0.25804897071889293,
 0.25336043053216756,
 0.24888825511418328,
 0.24460079675460009,
 0.2404775355753223,
 0.2364

## Evaluation (Hit@K)

We evaluate the model using a ranking-based metric.

For each user:
- Take one positive item
- Sample multiple negative items
- Rank all items using the model
- Check if the positive item is in the top-K

This is called **Hit@K**.

In [60]:
def hit_at_k(model, test_df, full_df, K=10, num_neg=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    hits = 0
    total = len(df) # We evaluate every positive interaction in the test set

    # Map interactions for quick lookup
    interacted_items = full_df.groupby('user_id')['item_id'].apply(set).to_dict()
    all_items = full_df['item_id'].unique()

    with torch.no_grad(): # Disable gradient calculation for speed/memory
        for _, row in df.iterrows():
            u = int(row['user_id'])
            pos_item = int(row['item_id'])

            # 1. Sample Negatives
            negatives = []
            while len(negatives) < num_neg:
                neg_item = np.random.choice(all_items)
                if neg_item not in interacted_items.get(u, set()):
                    negatives.append(neg_item)

            # 2. Prepare Tensors
            # We need a list of the 1 positive + 100 negatives
            item_list = [pos_item] + negatives
            user_tensor = torch.tensor([u] * (num_neg + 1)).to(device)
            item_tensor = torch.tensor(item_list).to(device)

            # 3. Get Scores
            scores = model(user_tensor, item_tensor)

            # 4. Rank and Check Hit
            # We want to see if the item at index 0 (the positive) is in the top K
            # torch.topk returns values and indices of the highest scores
            _, top_indices = torch.topk(scores, K)

            top_indices = top_indices.cpu().numpy()
            if 0 in top_indices:
                hits += 1

    return hits / total


In [61]:


# 3. Evaluate it
hit_rate = hit_at_k(MF_model, test_df, df, K=10, num_neg=100)
print(f"Hit@10: {hit_rate:.4f}")

Hit@10: 0.5579


If your score is low, try playing with the hyperparameters before moving on! Try sampling more negatives, or playing with the embedding dimensions.

Conversely, if your score is high, play with the values of K or number of negatives in evaluation (dec K, inc negatives).

## Neural Model: MLP-Based Recommender

Instead of using a dot product, we can learn a more flexible interaction function using a neural network.

Approach:
- Learn user and item embeddings
- **Concatenate** them
- Pass through a **Multi-Layer Perceptron (MLP)**

This allows the model to capture more complex relationships than simple similarity.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Concatenates them
3. Passes them through an MLP to produce a score

The choice of activation functions is upto you.

In [62]:
class MLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        # - Define item embedding layer
        # - Define MLP layers
        self.user_emb=nn.Embedding(num_users, emb_dim)
        self.item_emb=nn.Embedding(num_items, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * emb_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, user, item):
        # TODO:
        # - Get embeddings
        # - Concatenate user and item embeddings
        # - Pass through MLP
        # - Return a single score
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)
        concatenated = torch.cat([user_emb, item_emb], dim=1)
        return self.mlp(concatenated)[:,0]

## Train and Evaluate the MLP Model

Repeat the same steps as before:
- Train the model
- Evaluate on the test set

Compare its performance with the MF model.

In [63]:
# TODO:
# 1. Initialize MLP model
# 2. Train it
# 3. Evaluate it
MLP_model = MLP(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=16)
train(MLP_model, train_loader,valid_loader, epochs=300)
hit_rate = hit_at_k(MLP_model, test_df, df, K=10, num_neg=100)
print(f"Hit@10: {hit_rate:.4f}")


Epoch 1: Train Loss: 0.5922, Val Loss: 0.5329
Epoch 2: Train Loss: 0.4819, Val Loss: 0.4517
Epoch 3: Train Loss: 0.4260, Val Loss: 0.4243
Epoch 4: Train Loss: 0.4035, Val Loss: 0.4084
Epoch 5: Train Loss: 0.3918, Val Loss: 0.4050
Epoch 6: Train Loss: 0.3853, Val Loss: 0.4000
Epoch 7: Train Loss: 0.3807, Val Loss: 0.3975
Epoch 8: Train Loss: 0.3771, Val Loss: 0.4005
Epoch 9: Train Loss: 0.3743, Val Loss: 0.3984
Epoch 10: Train Loss: 0.3712, Val Loss: 0.3994
Epoch 11: Train Loss: 0.3654, Val Loss: 0.3996
Epoch 12: Train Loss: 0.3638, Val Loss: 0.4016
Epoch 13: Train Loss: 0.3623, Val Loss: 0.3980
Epoch 14: Train Loss: 0.3585, Val Loss: 0.3976
Epoch 15: Train Loss: 0.3577, Val Loss: 0.3977
Epoch 16: Train Loss: 0.3569, Val Loss: 0.4012
Epoch 17: Train Loss: 0.3549, Val Loss: 0.4001
Epoch 18: Train Loss: 0.3545, Val Loss: 0.4026
Epoch 19: Train Loss: 0.3541, Val Loss: 0.3997
Epoch 20: Train Loss: 0.3531, Val Loss: 0.4005
Epoch 21: Train Loss: 0.3529, Val Loss: 0.4022
Epoch 22: Train Loss: 

## Comparison & Analysis

You have now implemented:
- Matrix Factorization (MF)
- MLP-based recommender

---

### Compare the Models

- What score did each model achieve?
- Which model performed better?

---

### Think & Reflect

- Why might the MLP model outperform MF?
- In what cases might MF perform just as well or better?
- How does embedding size affect performance?
- Did you observe any signs of overfitting?

---

### Some Experiments

If you want to explore further:
- Try different embedding dimensions
- Change number of MLP layers
- Try different activation functions

---

## Bonus Task: Neural Collaborative Filtering (NCF)

In this task, you implemented:
- Matrix Factorization (MF)
- MLP-based recommender

The paper [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031) proposes combining both ideas.

---

### Idea

- MF captures **linear interactions**
- MLP captures **nonlinear interactions**

NCF combines both by:
1. Computing MF output
2. Computing MLP output
3. Combining them into a final prediction

---

### Your Task

Design a model that:
- Uses both MF and MLP components
- Combines their outputs
- Trains end-to-end

---

### Hints

- You can:
  - Concatenate MF and MLP representations
  - Or combine their final scores
- Think about:
  - Should embeddings be shared or separate?
  - How to balance both components?

---

Does combining both approaches improve performance over MF and MLP individually?

In [65]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, mf_dim=16, mlp_dim=16, layers=[64, 32, 16]):
        super(NCF, self).__init__()

        # Matrix Factorization (GMF) component
        self.mf_user_emb = nn.Embedding(num_users, mf_dim)
        self.mf_item_emb = nn.Embedding(num_items, mf_dim)

        # MLP component
        self.mlp_user_emb = nn.Embedding(num_users, mlp_dim)
        self.mlp_item_emb = nn.Embedding(num_items, mlp_dim)

        mlp_modules = []
        input_size = 2 * mlp_dim
        for output_size in layers:
            mlp_modules.append(nn.Linear(input_size, output_size))
            mlp_modules.append(nn.ReLU())
            input_size = output_size
        self.mlp_network = nn.Sequential(*mlp_modules)

        # Final Prediction Layer (Combines both outputs)
        self.prediction_layer = nn.Linear(mf_dim + layers[-1], 1)

    def forward(self, user, item):
        # GMF Branch
        mf_user = self.mf_user_emb(user)
        mf_item = self.mf_item_emb(item)
        mf_output = mf_user * mf_item # element-wise product

        # MLP Branch
        mlp_user = self.mlp_user_emb(user)
        mlp_item = self.mlp_item_emb(item)
        mlp_input = torch.cat([mlp_user, mlp_item], dim=1)
        mlp_output = self.mlp_network(mlp_input)

        # Concatenate GMF and MLP outputs
        combined = torch.cat([mf_output, mlp_output], dim=1)

        # Output layer
        return self.prediction_layer(combined).squeeze()

In [66]:
# 1. Initialize NCF model
NCF_model = NCF(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), mf_dim=16, mlp_dim=16)

# 2. Train it
print("Starting NCF training...")
train(NCF_model, train_loader, valid_loader, epochs=300)


Starting NCF training...
Epoch 1: Train Loss: 0.6080, Val Loss: 0.5405
Epoch 2: Train Loss: 0.4883, Val Loss: 0.4502
Epoch 3: Train Loss: 0.4280, Val Loss: 0.4153
Epoch 4: Train Loss: 0.4038, Val Loss: 0.4050
Epoch 5: Train Loss: 0.3916, Val Loss: 0.4010
Epoch 6: Train Loss: 0.3842, Val Loss: 0.4014
Epoch 7: Train Loss: 0.3785, Val Loss: 0.3985
Epoch 8: Train Loss: 0.3735, Val Loss: 0.3987
Epoch 9: Train Loss: 0.3686, Val Loss: 0.4019
Epoch 10: Train Loss: 0.3631, Val Loss: 0.4067
Epoch 11: Train Loss: 0.3542, Val Loss: 0.4057
Epoch 12: Train Loss: 0.3509, Val Loss: 0.4077
Epoch 13: Train Loss: 0.3479, Val Loss: 0.4137
Epoch 14: Train Loss: 0.3425, Val Loss: 0.4113
Epoch 15: Train Loss: 0.3406, Val Loss: 0.4125
Epoch 16: Train Loss: 0.3391, Val Loss: 0.4176
Epoch 17: Train Loss: 0.3361, Val Loss: 0.4203
Epoch 18: Train Loss: 0.3352, Val Loss: 0.4192
Epoch 19: Train Loss: 0.3344, Val Loss: 0.4151
Epoch 20: Train Loss: 0.3329, Val Loss: 0.4163
Epoch 21: Train Loss: 0.3324, Val Loss: 0.42

In [ ]:

# 3. Evaluate it
ncf_hit_rate = hit_at_k(NCF_model, test_df, df, K=10, num_neg=100)
print(f"NCF Hit@10: {ncf_hit_rate:.4f}")